In this notebook, we try to solve the one norm problem with cost function 

\begin{equation}
    J(u) = \|Au - b\|_2^2 + \alpha \|u\|_1
\end{equation}

by rewriting it into a least squares problem and using normal equations to solve the leastsquares problem. In each iteration, the least squares problem is

\begin{equation*}
    u_{k+1} = \arg\min_u \left\| \begin{bmatrix}
        A\\
        \sqrt{\alpha} W^{(k)}
    \end{bmatrix}u - \begin{bmatrix}
        b\\
        0
    \end{bmatrix}\right\|_2^2
\end{equation*}

with $W^{(k)} = \text{diag}\{u^{(k)^{-1/2}}\}$

The normal equations are given as 

\begin{equation}
    (A^TA + \alpha W)^{-1}A^Tb = u
\end{equation}

In [1]:
# import necessary packages

# CIL imports
from cil.framework import ImageGeometry, AcquisitionGeometry, BlockDataContainer, ImageData

# From cil.plugins import TomoPhantom
from phantominator import shepp_logan

from cil.optimisation.functions import LeastSquares

# For display
from cil.utilities.display import show2D, show1D, show_geometry

# ASTRA imports
from cil.plugins.astra.operators import ProjectionOperator

from cil.optimisation.algorithms import CGLS, GD

# External imports
import numpy as np
import matplotlib.pyplot as plt
import logging

from cil.optimisation.operators import BlockOperator, DiagonalOperator

import scipy as sp

In [2]:
# setting pixels and angles
n_pixels = 50
n_angles = 40

## Setting up phantom system and defaults

In [3]:
# Set logging level for CIL processors:
logging.basicConfig(level=logging.WARNING)
cil_log_level = logging.getLogger('cil.processors')
cil_log_level.setLevel(logging.INFO)

# Setting background defaults
cmap = "rainbow"
device = "cpu"

In [4]:
# Angles
angles = np.linspace(0, 180, n_angles, endpoint=False, dtype=np.float32)

# Setup acquisition geometry
ag = AcquisitionGeometry.create_Parallel2D()\
                            .set_angles(angles)\
                            .set_panel(n_pixels, pixel_size=1/n_pixels)

# Setup image geometry
ig = ImageGeometry(voxel_num_x=n_pixels, 
                   voxel_num_y=n_pixels, 
                   voxel_size_x=1/n_pixels, 
                   voxel_size_y=1/n_pixels)

# Get phantom
phantom = ImageData(np.flip(shepp_logan(n_pixels)), geometry = ig)

# Create projection operator using Astra-Toolbox.
A = ProjectionOperator(ig, ag, device)

# Create an acquisition data (numerically)
sino = A.direct(phantom)

c:\Users\femke\anaconda3\envs\cil_demos_cpu\Lib\site-packages\cil\framework\data_container.py:112: UserWarning: Over-riding geometry.dtype with data.dtype
  warnings.warn("Over-riding geometry.dtype with data.dtype", UserWarning)


## Setting up normal equations

In [35]:
def onenorm_normal(max_outer, max_inner, initial, A, sino, alpha):
    uk = initial.copy()

    for k in range(max_outer):
        
        # setting up term \alpha W
        w_imagedata = (uk + 1e-8) ** (-2) # add very small number to avoid division by zero
        w_imagedata = w_imagedata ** (1/4)
        w_operator = alpha * DiagonalOperator(w_imagedata)

        # creating lefthand side of normal equation
        normal = A + w_operator
        sp_normal = sp.sparse.linalg.aslinearoperator(normal)

        # creating righthand side of normal equation
        b = A.direct(sino)

        # inner loop to find next iteration
        scipy_normal = sp.sparse.linalg.LinearOperator(normal)
        uk = sp.sparse.linalg.cg(normal, b)

    return uk

In [36]:
x0 = ig.allocate(0)
test = onenorm_normal(10, 5, x0, A, sino, 1)

TypeError: type not understood

Using CGLS to solve normal equations

In [37]:
def onenorm_normal_cgls(max_outer, max_inner, initial, A, sino, alpha):
    uk = initial.copy()

    for k in range(max_outer):
        
        # setting up term \alpha W
        w_imagedata = (uk + 1e-8) ** (-2) # add very small number to avoid division by zero
        w_imagedata = w_imagedata ** (1/4)
        w_operator = alpha * DiagonalOperator(w_imagedata)

        # creating lefthand side of normal equation
        normal = A + w_operator

        # creating righthand side of normal equation
        b = A.adjoint(sino)

        print(np.shape(normal), np.shape(b))
        # inner loop to find next iteration
        cgls = CGLS(initial=uk, operator=normal, data=b, update_objective_interval=10)
        cgls.run(max_inner, verbose=1)

        uk = cgls.solution

    return uk

In [38]:
test2 = onenorm_normal_cgls(10, 5, x0, A, sino, 1)

() (50, 50)


ValueError: Wrong size for data memory: out (40, 50) x2 (50, 50) expected (40, 50)